# Shared Holdout Splits for 5-Model Evaluation

This notebook reproduces the **same leave-one-fake-out split protocol** used in the WavLM ablation notebook and evaluates 5 models on those exact holdout test sets.

Experiments:
- holdout_crikk
- holdout_lstm
- holdout_gemini
- holdout_baglafake

Model slots (editable):
- lstm
- wavenet
- wave2vec
- wavlm_aasist
- xlsr_aasist

In [ ]:
import os
import re
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
IS_KAGGLE = os.path.exists('/kaggle/input')
OUTPUT_ROOT = '/kaggle/working/shared_holdout_5model_eval' if IS_KAGGLE else './shared_holdout_5model_eval'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

REAL_FOLDERS = ['banglafake_real', 'lstm_real']
FAKE_FOLDERS = ['baglafake_deepfake', 'crikk_deepfake', 'gemini_deepfake', 'lstm_deepfake']
ALL_REQUIRED_FOLDERS = REAL_FOLDERS + FAKE_FOLDERS

EXPERIMENTS = [
    {'name': 'holdout_crikk', 'holdout_fake': 'crikk_deepfake'},
    {'name': 'holdout_lstm', 'holdout_fake': 'lstm_deepfake'},
    {'name': 'holdout_gemini', 'holdout_fake': 'gemini_deepfake'},
    {'name': 'holdout_baglafake', 'holdout_fake': 'baglafake_deepfake'},
]

print('Output root:', os.path.abspath(OUTPUT_ROOT))
print('Experiments:', [e['name'] for e in EXPERIMENTS])

In [ ]:
def _norm_name(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())

def _resolve_folder_aliases(root_dir, expected_names):
    if not os.path.isdir(root_dir):
        return None

    children = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    children_norm = {_norm_name(d): d for d in children}

    aliases = {
        'baglafake_deepfake': ['baglafake_deepfake', 'banglafake_deepfake'],
        'banglafake_real': ['banglafake_real', 'baglafake_real'],
        'crikk_deepfake': ['crikk_deepfake'],
        'gemini_deepfake': ['gemini_deepfake'],
        'lstm_deepfake': ['lstm_deepfake'],
        'lstm_real': ['lstm_real'],
    }

    resolved = {}
    for exp in expected_names:
        candidates = aliases.get(exp, [exp])
        found = None
        for c in candidates:
            c_norm = _norm_name(c)
            if c_norm in children_norm:
                found = os.path.join(root_dir, children_norm[c_norm])
                break
        if found is None:
            return None
        resolved[exp] = found

    return resolved

def resolve_data_root(required_folders):
    candidates = [
        './Data',
        './Bangla Audio DeepFake/Data',
        '/kaggle/working/Data',
        '/kaggle/input/Data',
    ]

    for c in candidates:
        mapped = _resolve_folder_aliases(c, required_folders)
        if mapped is not None:
            return os.path.abspath(c), mapped

    if os.path.isdir('/kaggle/input'):
        for ds in sorted(glob.glob('/kaggle/input/*')):
            mapped = _resolve_folder_aliases(ds, required_folders)
            if mapped is not None:
                return os.path.abspath(ds), mapped

            for root, dirs, _ in os.walk(ds):
                rel = os.path.relpath(root, ds)
                depth = 0 if rel == '.' else rel.count(os.sep) + 1
                if depth > 3:
                    dirs[:] = []
                    continue
                if os.path.basename(root).lower() == 'data':
                    mapped = _resolve_folder_aliases(root, required_folders)
                    if mapped is not None:
                        return os.path.abspath(root), mapped

    raise FileNotFoundError('Could not locate Data folder with required subfolders.')

DATA_ROOT, RESOLVED_DATA_FOLDERS = resolve_data_root(ALL_REQUIRED_FOLDERS)
print('Data root:', DATA_ROOT)

In [ ]:
def build_dataset_df(data_root):
    records = []

    for folder in REAL_FOLDERS:
        folder_path = RESOLVED_DATA_FOLDERS[folder]
        wav_files = sorted(glob.glob(os.path.join(folder_path, '*.wav')))
        for fp in wav_files:
            records.append({
                'filepath': os.path.abspath(fp),
                'label': 0,
                'label_name': 'Real',
                'source': folder,
            })

    for folder in FAKE_FOLDERS:
        folder_path = RESOLVED_DATA_FOLDERS[folder]
        wav_files = sorted(glob.glob(os.path.join(folder_path, '*.wav')))
        for fp in wav_files:
            records.append({
                'filepath': os.path.abspath(fp),
                'label': 1,
                'label_name': 'Fake',
                'source': folder,
            })

    if len(records) == 0:
        raise RuntimeError(f'No wav files found under resolved paths from {data_root}')

    return pd.DataFrame(records)

def stratified_split(df, train_ratio=0.70, val_ratio=0.15, seed=42):
    """Exact WavLM-style stratified split with optional spill return."""
    assert abs((train_ratio + val_ratio) - 1.0) < 1e-8, 'train_ratio + val_ratio must be 1.0'

    train_parts, val_parts, spill_parts = [], [], []

    for lbl in sorted(df['label'].unique()):
        sub = df[df['label'] == lbl].sample(frac=1, random_state=seed).reset_index(drop=True)
        n = len(sub)

        n_train = int(round(n * train_ratio))
        n_val = int(round(n * val_ratio))
        if n_train + n_val > n:
            n_val = n - n_train

        train_parts.append(sub.iloc[:n_train])
        val_parts.append(sub.iloc[n_train:n_train + n_val])
        if n_train + n_val < n:
            spill_parts.append(sub.iloc[n_train + n_val:])

    train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    spill_df = pd.concat(spill_parts, ignore_index=True) if spill_parts else pd.DataFrame(columns=df.columns)

    return train_df, val_df, spill_df

def split_leave_one_fake_out_df(df_all, holdout_fake, val_ratio=0.15, seed=42):
    fake_df = df_all[df_all['label'] == 1].reset_index(drop=True)
    real_df = df_all[df_all['label'] == 0].reset_index(drop=True)

    fake_test = fake_df[fake_df['source'] == holdout_fake].copy().reset_index(drop=True)
    fake_train = fake_df[fake_df['source'] != holdout_fake].copy().reset_index(drop=True)

    if len(fake_test) == 0 or len(fake_train) == 0:
        raise ValueError(f'Invalid holdout split for {holdout_fake}: empty fake train/test set.')

    fake_train_ratio = len(fake_train) / float(len(fake_train) + len(fake_test))

    real_idx = np.arange(len(real_df))
    rng = np.random.default_rng(seed)
    rng.shuffle(real_idx)

    n_real_train = int(round(len(real_df) * fake_train_ratio))
    n_real_train = max(1, min(len(real_df) - 1, n_real_train))

    real_train = real_df.iloc[real_idx[:n_real_train]].copy().reset_index(drop=True)
    real_test = real_df.iloc[real_idx[n_real_train:]].copy().reset_index(drop=True)

    train_pool = pd.concat([fake_train, real_train], ignore_index=True)
    holdout_test_df = pd.concat([fake_test, real_test], ignore_index=True)

    # Exact WavLM behavior: split with spill and fold spill back into train
    train_df, val_df, spill_df = stratified_split(train_pool, train_ratio=1.0 - val_ratio, val_ratio=val_ratio, seed=seed)
    if len(spill_df) > 0:
        train_df = pd.concat([train_df, spill_df], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

    holdout_test_df = holdout_test_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    return train_df, val_df, holdout_test_df

In [ ]:
df_all = build_dataset_df(DATA_ROOT)
print('Total samples:', len(df_all))
display(df_all.groupby(['source', 'label_name']).size().unstack(fill_value=0))

split_root = Path(OUTPUT_ROOT) / 'shared_splits'
split_root.mkdir(parents=True, exist_ok=True)

split_summaries = []
for exp in EXPERIMENTS:
    exp_name = exp['name']
    holdout_fake = exp['holdout_fake']

    train_df, val_df, holdout_test_df = split_leave_one_fake_out_df(df_all, holdout_fake=holdout_fake, val_ratio=0.15, seed=SEED)

    exp_dir = split_root / exp_name / 'manifests'
    exp_dir.mkdir(parents=True, exist_ok=True)

    train_df.to_csv(exp_dir / 'train_manifest.csv', index=False)
    val_df.to_csv(exp_dir / 'val_manifest.csv', index=False)
    holdout_test_df.to_csv(exp_dir / 'holdout_test_manifest.csv', index=False)

    split_summaries.append({
        'experiment': exp_name,
        'holdout_fake_source': holdout_fake,
        'train_size': len(train_df),
        'val_size': len(val_df),
        'holdout_test_size': len(holdout_test_df),
        'train_real': int((train_df['label'] == 0).sum()),
        'train_fake': int((train_df['label'] == 1).sum()),
        'holdout_test_real': int((holdout_test_df['label'] == 0).sum()),
        'holdout_test_fake': int((holdout_test_df['label'] == 1).sum()),
    })

split_summary_df = pd.DataFrame(split_summaries)
split_summary_df.to_csv(split_root / 'split_summary.csv', index=False)
display(split_summary_df)
print('Shared manifests saved to:', split_root)

## Direct WavLM Checkpoint Evaluation

This section evaluates **holdout sets directly using WavLM checkpoints** generated by the WavLM ablation notebook.

Expected checkpoint structure:
- `<WAVLM_OUTPUT_ROOT>/<experiment>/models/best_model.pt`
for experiments: `holdout_crikk`, `holdout_lstm`, `holdout_gemini`, `holdout_baglafake`.

The code below:
1. loads each experiment checkpoint,
2. reads that experiment's `holdout_test_manifest.csv`,
3. runs inference on those files,
4. computes holdout metrics,
5. saves `wavlm_direct_holdout_results.csv`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from transformers import WavLMModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAMPLE_RATE = 16000
AUDIO_DURATION = 5
MAX_AUDIO_LEN = SAMPLE_RATE * AUDIO_DURATION

WAVLM_MODEL_NAME = 'microsoft/wavlm-large'
WAVLM_DIM = 1024
WAVLM_FREEZE_LAYERS = 12
AASIST_HIDDEN = 160
AASIST_EMB_DIM = 128
AASIST_NUM_HEADS = 4
AASIST_NUM_GAT_LAYERS = 2
NUM_SPECTRAL_NODES = 32
DROPOUT = 0.3
OC_MARGIN = 0.5
OC_SCALE = 10.0

def binary_prf(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    f1 = float((2 * precision * recall) / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def binary_auc(y_true, y_score):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    pos = np.sum(y_true == 1)
    neg = np.sum(y_true == 0)
    if pos == 0 or neg == 0:
        return np.nan

    order = np.argsort(-y_score, kind='mergesort')
    y_true = y_true[order]
    y_score = y_score[order]

    distinct = np.where(np.diff(y_score))[0]
    idx = np.r_[distinct, y_true.size - 1]
    tps = np.cumsum(y_true)[idx]
    fps = (1 + idx) - tps
    tps = np.r_[0, tps]
    fps = np.r_[0, fps]
    tpr = tps / pos
    fpr = fps / neg
    return float(np.trapz(tpr, fpr))

class ModifiedAASIST(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=160, emb_dim=128, num_heads=4, num_gat_layers=2, num_spectral_nodes=32, max_seq_len=512, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pos_enc = nn.Parameter(torch.randn(1, max_seq_len, hidden_dim) * 0.02)
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1, groups=hidden_dim),
            nn.Conv1d(hidden_dim, hidden_dim, 1),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )
        self.spectral_basis = nn.Parameter(torch.randn(num_spectral_nodes, hidden_dim) * 0.02)
        self.spectral_proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.drop = nn.Dropout(dropout)

        self.gat_layers = nn.ModuleList()
        self.gat_norms1 = nn.ModuleList()
        self.gat_ffn = nn.ModuleList()
        self.gat_norms2 = nn.ModuleList()
        for _ in range(num_gat_layers):
            self.gat_layers.append(nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True))
            self.gat_norms1.append(nn.LayerNorm(hidden_dim))
            self.gat_ffn.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 4, hidden_dim),
                nn.Dropout(dropout),
            ))
            self.gat_norms2.append(nn.LayerNorm(hidden_dim))

        self.readout_query = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        self.readout_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.readout_norm = nn.LayerNorm(hidden_dim)
        self.output_fc = nn.Sequential(
            nn.Linear(hidden_dim, emb_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, x):
        bsz, tlen, _ = x.shape
        h = self.input_proj(x) + self.pos_enc[:, :tlen, :]
        h = self.drop(h)
        h = h + self.temporal_conv(h.transpose(1, 2)).transpose(1, 2)
        s_nodes = self.spectral_basis.unsqueeze(0).expand(bsz, -1, -1)
        w = F.softmax(torch.bmm(s_nodes, h.transpose(1, 2)) / math.sqrt(h.size(-1)), dim=-1)
        spectral = self.spectral_proj(torch.bmm(w, h))
        nodes = torch.cat([h, spectral], dim=1)

        for gat, n1, ffn, n2 in zip(self.gat_layers, self.gat_norms1, self.gat_ffn, self.gat_norms2):
            res = nodes
            out, _ = gat(n1(nodes), n1(nodes), n1(nodes))
            nodes = res + self.drop(out)
            nodes = nodes + ffn(n2(nodes))

        q = self.readout_query.expand(bsz, -1, -1)
        out, attn = self.readout_attn(q, nodes, nodes)
        emb = self.readout_norm(out).squeeze(1)
        return self.output_fc(emb), attn

class OCSoftmaxLoss(nn.Module):
    def __init__(self, feat_dim=128, margin=0.5, scale=10.0):
        super().__init__()
        self.w = nn.Parameter(torch.empty(feat_dim))
        nn.init.normal_(self.w, mean=0, std=0.01)
        self.margin = margin
        self.scale = scale

    def score(self, x):
        return F.cosine_similarity(
            F.normalize(x, dim=1),
            F.normalize(self.w.unsqueeze(0), dim=1).expand_as(x),
            dim=1,
        )

class BanglaDeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained(WAVLM_MODEL_NAME)
        self._freeze_wavlm()
        self.aasist = ModifiedAASIST(
            input_dim=WAVLM_DIM, hidden_dim=AASIST_HIDDEN, emb_dim=AASIST_EMB_DIM,
            num_heads=AASIST_NUM_HEADS, num_gat_layers=AASIST_NUM_GAT_LAYERS,
            num_spectral_nodes=NUM_SPECTRAL_NODES, dropout=DROPOUT
        )
        self.oc_softmax = OCSoftmaxLoss(feat_dim=AASIST_EMB_DIM, margin=OC_MARGIN, scale=OC_SCALE)

    def _freeze_wavlm(self):
        self.wavlm.feature_extractor._freeze_parameters()
        for p in self.wavlm.feature_projection.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.wavlm.encoder.layers):
            if i < WAVLM_FREEZE_LAYERS:
                for p in layer.parameters():
                    p.requires_grad = False

    def forward(self, audio):
        feats = self.wavlm(input_values=audio).last_hidden_state
        emb, _ = self.aasist(feats)
        return self.oc_softmax.score(emb)

def load_audio_for_model(filepath):
    audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, duration=AUDIO_DURATION)
    if len(audio) < MAX_AUDIO_LEN:
        audio = np.pad(audio, (0, MAX_AUDIO_LEN - len(audio)))
    else:
        audio = audio[:MAX_AUDIO_LEN]
    audio = (audio - audio.mean()) / (audio.std() + 1e-7)
    return audio.astype(np.float32)

@torch.no_grad()
def predict_scores(model, filepaths, device):
    scores = []
    for fp in filepaths:
        wav = load_audio_for_model(fp)
        x = torch.from_numpy(wav).unsqueeze(0).to(device)
        s = model(x)
        scores.append(float(s[0].detach().cpu().item()))
    return np.array(scores, dtype=np.float32)

def resolve_wavlm_output_root():
    candidates = [
        Path('/kaggle/working/wavlm_aasist_outputs'),
        Path('./wavlm_aasist_outputs'),
        Path('../wavlm_aasist_outputs'),
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c.resolve()

    cwd = Path.cwd()
    found = list(cwd.rglob('wavlm_aasist_outputs'))
    found = [p for p in found if p.is_dir()]
    if found:
        return found[0].resolve()

    raise FileNotFoundError('Could not find wavlm_aasist_outputs directory.')

WAVLM_OUTPUT_ROOT = resolve_wavlm_output_root()
print('Using WavLM output root:', WAVLM_OUTPUT_ROOT)

results = []
for exp in EXPERIMENTS:
    exp_name = exp['name']
    holdout_fake = exp['holdout_fake']

    manifest_path = split_root / exp_name / 'manifests' / 'holdout_test_manifest.csv'
    ckpt_path = WAVLM_OUTPUT_ROOT / exp_name / 'models' / 'best_model.pt'

    if not manifest_path.exists():
        results.append({
            'experiment': exp_name,
            'model': 'wavlm_aasist',
            'status': 'missing_holdout_manifest',
            'manifest_path': str(manifest_path),
        })
        continue

    if not ckpt_path.exists():
        results.append({
            'experiment': exp_name,
            'model': 'wavlm_aasist',
            'status': 'missing_checkpoint',
            'checkpoint_path': str(ckpt_path),
        })
        continue

    holdout_df = pd.read_csv(manifest_path)
    holdout_df['filepath'] = holdout_df['filepath'].astype(str).map(os.path.abspath)

    model = BanglaDeepfakeDetector().to(DEVICE)
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    threshold = float(ckpt.get('eer_threshold', 0.0))
    y_true = holdout_df['label'].astype(int).values
    scores = predict_scores(model, holdout_df['filepath'].tolist(), DEVICE)
    y_pred = (scores <= threshold).astype(int)

    acc = float(np.mean(y_true == y_pred))
    prec, rec, f1 = binary_prf(y_true, y_pred)
    auc = binary_auc(y_true, -scores)

    pred_df = holdout_df.copy()
    pred_df['score'] = scores
    pred_df['pred_label'] = y_pred
    pred_df['eer_threshold'] = threshold

    out_pred_dir = Path(OUTPUT_ROOT) / 'wavlm_direct_predictions' / exp_name
    out_pred_dir.mkdir(parents=True, exist_ok=True)
    pred_path = out_pred_dir / 'holdout_test_predictions.csv'
    pred_df.to_csv(pred_path, index=False)

    results.append({
        'experiment': exp_name,
        'holdout_fake_source': holdout_fake,
        'model': 'wavlm_aasist',
        'status': 'ok',
        'holdout_test_size': int(len(holdout_df)),
        'holdout_test_accuracy': acc,
        'holdout_test_precision': float(prec),
        'holdout_test_recall': float(rec),
        'holdout_test_f1': float(f1),
        'holdout_test_auc': float(auc) if not np.isnan(auc) else np.nan,
        'eer_threshold': threshold,
        'checkpoint_path': str(ckpt_path),
        'prediction_file': str(pred_path),
    })

results_df = pd.DataFrame(results)
results_csv = Path(OUTPUT_ROOT) / 'wavlm_direct_holdout_results.csv'
results_df.to_csv(results_csv, index=False)

print('Saved:', results_csv)
display(results_df)

## Identical Split Guarantee

You get **identical holdout test splits** across models if all of the following are true:
1. You generate splits once with this notebook (fixed `SEED`) and do not regenerate with changed data.
2. All 5 models read the same `holdout_test_manifest.csv` for each experiment.
3. No model silently drops files (failed decode, missing file, etc.).
4. Evaluation merges by normalized absolute `filepath` and requires full match (`matched_size == gt_size`).

If any model has `status != ok`, splits are no longer strictly identical for comparison.